In [37]:
import os
import json
from zipfile import ZipFile
from PIL import Image

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2

In [38]:
import random
random.seed(0)

import numpy as np
np.random.seed(0)

import tensorflow as tf
tf.random.set_seed(0)

3: Dataset Path

In [39]:
# Dataset Path - LOCAL
base_dir = r"C:\Users\PC\Downloads\Plant_disease_classifier-main-well\Plant_disease_classifier-main\PlantVillage"

# verify
print(f"Base directory: {base_dir}")
print(f"Classes found: {os.listdir(base_dir)}")
print(f"Number of classes: {len(os.listdir(base_dir))}")

Base directory: C:\Users\PC\Downloads\Plant_disease_classifier-main-well\Plant_disease_classifier-main\PlantVillage
Classes found: ['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___healthy', 'Potato___Late_blight', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_healthy', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_mosaic_virus', 'Tomato__Tomato_YellowLeaf__Curl_Virus']
Number of classes: 15


4: Image Parameters

In [40]:
img_size = 224
batch_size = 32

5: Data Generators

In [41]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    base_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation'
)

Found 16516 images belonging to 15 classes.
Found 4122 images belonging to 15 classes.


6: Model Architecture

In [42]:
# Model using a MobileNetV2 backbone (pretrained on ImageNet)
# the base is frozen for feature extraction; only the head will be trained
base = MobileNetV2(input_shape=(img_size, img_size, 3),
                   include_top=False,
                   weights='imagenet')
base.trainable = False  # freeze the convolutional base

model = models.Sequential([
    base,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(train_generator.num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 mobilenetv2_1.00_224 (Funct  (None, 7, 7, 1280)       2257984   
 ional)                                                          
                                                                 
 global_average_pooling2d_3   (None, 1280)             0         
 (GlobalAveragePooling2D)                                        
                                                                 
 dense_6 (Dense)             (None, 256)               327936    
                                                                 
 dropout_3 (Dropout)         (None, 256)               0         
                                                                 
 dense_7 (Dense)             (None, 15)                3855      
                                                                 
Total params: 2,589,775
Trainable params: 331,791
Non-

7: Training

In [44]:
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=validation_generator
)

Epoch 1/10
517/517 [==============================] - 108s 209ms/step - loss: 0.6698 - accuracy: 0.7789 - val_loss: 0.4353 - val_accuracy: 0.8557
Epoch 2/10
517/517 [==============================] - 101s 196ms/step - loss: 0.5780 - accuracy: 0.8079 - val_loss: 0.3989 - val_accuracy: 0.8680
Epoch 3/10
517/517 [==============================] - 98s 190ms/step - loss: 0.5249 - accuracy: 0.8224 - val_loss: 0.3792 - val_accuracy: 0.8709
Epoch 4/10
517/517 [==============================] - 99s 191ms/step - loss: 0.5033 - accuracy: 0.8300 - val_loss: 0.3635 - val_accuracy: 0.8748
Epoch 5/10
517/517 [==============================] - 100s 193ms/step - loss: 0.4804 - accuracy: 0.8388 - val_loss: 0.3439 - val_accuracy: 0.8874
Epoch 6/10
517/517 [==============================] - 98s 190ms/step - loss: 0.4612 - accuracy: 0.8445 - val_loss: 0.3426 - val_accuracy: 0.8782
Epoch 7/10
517/517 [==============================] - 98s 190ms/step - loss: 0.4524 - accuracy: 0.8454 - val_loss: 0.3115 - val

8: Save Model

In [45]:
# Save model to app/trained_model folder (relative path)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
output_dir = os.path.join(project_root, 'app', 'trained_model')

os.makedirs(output_dir, exist_ok=True)
model_path = os.path.join(output_dir, 'plant_disease_prediction_model.keras')
model.save(model_path)
print(f"Model saved successfully at {model_path}")

Model saved successfully at c:\Users\PC\Downloads\Plant_disease_classifier-main-well\Plant_disease_classifier-main\app\trained_model\plant_disease_prediction_model.keras


9: Save Class Indices

In [46]:
# Create a mapping from class indices to class names
class_indices = {v: k for k, v in train_generator.class_indices.items()}

# Save to app folder (relative path)
config_dir = os.path.join(project_root, 'app')
with open(os.path.join(config_dir, 'class_indices.json'), 'w') as f:
    json.dump(class_indices, f)
print("Class indices saved successfully!")

Class indices saved successfully!
